In [7]:
pip install openmeteo-requests requests-cache retry-requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.7/207.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 711.0/711.0 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.1/394.1 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.3 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19


In [10]:
from datetime import datetime
import openmeteo_requests

# ITERATORS

class IncreaseSpeed:
    def __init__(self, current_speed, max_speed, step=10):
        self.speed = current_speed
        self.max_speed = max_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.speed >= self.max_speed:
            raise StopIteration
        self.speed = min(self.speed + self.step, self.max_speed)
        return self.speed


class DecreaseSpeed:
    def __init__(self, current_speed, min_speed=0, step=10):
        # Ensure min_speed is never negative
        self.min_speed = max(0, min_speed)
        self.speed = current_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.speed <= self.min_speed:
            raise StopIteration
        self.speed = max(self.speed - self.step, self.min_speed)
        return self.speed


# CAR

class Car:
    cars_on_road = 0

    def __init__(self, max_speed, current_speed=0):
        self.max_speed = max_speed
        self.current_speed = current_speed

        if current_speed == 0:
            self.state = False  # False = in parking
        else:
            self.state = True   # True = on road
            Car.cars_on_road += 1

    def accelerate(self, upper_border=None, step=10):
        old_speed = self.current_speed

        # Bring car to road if parked
        if not self.state:
            self.state = True
            Car.cars_on_road += 1

        iterator = IncreaseSpeed(self.current_speed, self.max_speed, step)

        if upper_border is not None:
            for speed in iterator:
                print("INFO: Speed increases by 10")
                self.current_speed = speed
                if self.current_speed >= upper_border:
                    break
        else:
            try:
                print("INFO: Speed increases by 10")
                self.current_speed = next(iterator)
            except StopIteration:
                pass

        print(f"INFO: The speed of this car has been increased from {old_speed} to {self.current_speed}")

    def brake(self, lower_border=None, step=10):
        old_speed = self.current_speed

        # Validate lower_border to prevent negative values
        if lower_border is not None:
            lower_border = max(0, lower_border)
        else:
            lower_border = 0

        # Only create iterator if we can actually decrease speed
        if lower_border < self.current_speed:
            iterator = DecreaseSpeed(self.current_speed, lower_border, step)

            for speed in iterator:
                print("INFO: Speed decreases by 10")
                self.current_speed = speed
                if self.current_speed <= lower_border:
                    break

            print(f"INFO: The speed of this car has been decreased from {old_speed} to {self.current_speed}")
        else:
            # No speed decrease needed
            print(f"INFO: The speed of this car has been decreased from {old_speed} to {self.current_speed}")

    def parking(self):
        # Handle case when already at 0
        if self.current_speed > 0:
            old_speed = self.current_speed
            self.current_speed = 0
            print(f"INFO: The speed of this car has been decreased from {old_speed} to 0")
        else:
            print("INFO: The speed of this car has been decreased from 0 to 0")

        if self.state:
            print("Parking the car...")
            self.state = False
            Car.cars_on_road -= 1

    @classmethod
    def total_cars(cls):
        return cls.cars_on_road

    @staticmethod
    def show_weather():
        try:
            openmeteo = openmeteo_requests.Client()

            url = "https://api.open-meteo.com/v1/forecast"
            params = {
                "latitude": 59.9386,
                "longitude": 30.3141,
                "current": ["temperature_2m", "apparent_temperature", "rain", "wind_speed_10m"],
                "wind_speed_unit": "ms",
                "timezone": "Europe/Moscow"
            }

            responses = openmeteo.weather_api(url, params=params)

            if responses:
                response = responses[0]
                current = response.Current()

                print(f"Current temperature: {round(current.Variables(0).Value(), 1)} C")
                print(f"Current apparent_temperature: {round(current.Variables(1).Value(), 1)} C")
                print(f"Current rain: {current.Variables(2).Value()} mm")
                print(f"Current wind_speed: {round(current.Variables(3).Value(), 1)} m/s")
            else:
                print("Unable to fetch weather data")

        except Exception as e:
            print(f"Weather data unavailable: {e}")

In [11]:
car1 = Car(100, 20)
car2 = Car(60, 30)
car3 = Car(100, 0)
print(f"Total cars on road: {Car.total_cars()}")

Total cars on road: 2


In [12]:
car1.accelerate(100)

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 20 to 100


In [13]:
car2.accelerate(50)

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 30 to 50


In [14]:
print("Speed of car 1:", car1.current_speed)

Speed of car 1: 100


In [15]:
print("Speed of car 2:", car2.current_speed)

Speed of car 2: 50


In [16]:
car1.brake(10)

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 100 to 10


In [17]:
car2.brake(0)
print("Total cars on road:", Car.total_cars())
car2.parking()
print("Total cars on road:", Car.total_cars())

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 50 to 0
Total cars on road: 2
INFO: The speed of this car has been decreased from 0 to 0
Parking the car...
Total cars on road: 1


In [18]:
car3.accelerate(80)# car3 is now on the road
car3.show_weather()
print("Total cars on road:", Car.total_cars())

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 0 to 80
Current temperature: 2.2 C
Current apparent_temperature: -2.3 C
Current rain: 0.0 mm
Current wind_speed: 4.9 m/s
Total cars on road: 2


In [19]:
car2.accelerate(10) # # car2 goes from parking on the road
print("Total cars on road:", Car.total_cars())

INFO: Speed increases by 10
INFO: The speed of this car has been increased from 0 to 10
Total cars on road: 3


In [23]:
Car.show_weather()

Current temperature: 2.2 C
Current apparent_temperature: -2.4 C
Current rain: 0.0 mm
Current wind_speed: 4.9 m/s
